# SEED-VII EEG-Conformer 端到端流水线

**OOM 终极修复 + 过拟合缓解模式**

内存策略：
1. 一次性将压缩 npz 转换为 X.npy + meta.npz（mmap 格式）
2. 训练时 X 用内存映射加载（零 RAM），只把需要的子集拷贝进 RAM
3. 单被试训练只需 ~600MB RAM（vs 全量 ~12GB）


## 0. 环境与路径

In [ ]:
import os, sys, subprocess, json, pathlib

REPO = pathlib.Path('/mnt/workspace/EEG_encoder_version2').resolve()
assert (REPO / 'scripts' / 'preprocess_to_npz.py').is_file(), f'Repo not found at {REPO}'
print('REPO =', REPO)
sys.path.insert(0, str(REPO))

# ====== ModelScope ======
MS_DATASET = 'DEREKVERSE/SEED-VII'
MS_REVISION = 'master'
MS_TOKEN = os.environ.get('MODELSCOPE_API_TOKEN', 'ms-2460c377-dcfc-4cb5-86d5-635cfa6ea9e2')

# ====== 本地路径 ======
WORK = pathlib.Path('/mnt/workspace/seed_vii_work')
WORK.mkdir(parents=True, exist_ok=True)
MAT_LOCAL_DIR = str(WORK / 'mat_files')
SAVE_INFO_LOCAL = str(WORK / 'save_info')
NPZ_OUT = str(WORK / 'preprocessed' / 'seed_vii.npz')
RUN_DIR = str(WORK / 'runs_seed_vii')
ENC_OUT = str(WORK / 'encoded' / 'embeddings.npz')
NPZ_REPO_PATH = 'artifacts/preprocessed/seed_vii.npz'

# ====== 过拟合缓解配置 ======
FREEZE_INTENSITY = True
TRAIN_SUBJECTS = '1'       # 只用被试1训练
VAL_SUBJECTS = '2,3'       # 被试2,3验证
TEST_SUBJECTS = '4,5'      # 被试4,5测试

def run(cmd, env=None):
    print('$', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=str(REPO), env={**os.environ, **(env or {})})
    if res.returncode != 0:
        raise SystemExit(f'cmd failed with code {res.returncode}')

PY = sys.executable
print('python =', PY)

In [ ]:
import os
os.environ['MODELSCOPE_API_TOKEN'] = 'ms-2460c377-dcfc-4cb5-86d5-635cfa6ea9e2'
from modelscope.hub.api import HubApi
HubApi().login(os.environ['MODELSCOPE_API_TOKEN'])
print('login done')

## 1. 下载 MAT + 流式预处理 → npz

In [ ]:
from pathlib import Path
from src.ms_download import download_files, list_dataset_files, filter_files, login_if_token

login_if_token(MS_TOKEN)
Path(MAT_LOCAL_DIR).mkdir(parents=True, exist_ok=True)
Path(SAVE_INFO_LOCAL).mkdir(parents=True, exist_ok=True)

all_files = list_dataset_files(MS_DATASET, revision=MS_REVISION, token=MS_TOKEN)
mat_files = [f for f in all_files if f['Path'].lower().endswith('.mat') and 'eeg_preprocessed' in f['Path'].lower()]
save_files = [f for f in all_files if f['Path'].lower().endswith('_save_info.csv')]
if not mat_files: mat_files = filter_files(all_files, include=['EEG_preprocessed/*.mat', '*.mat'])
if not save_files: save_files = filter_files(all_files, include=['save_info/*_save_info.csv'])

mat_paths = [f['Path'] for f in mat_files]
save_paths = [f['Path'] for f in save_files]
print(f'Found {len(mat_paths)} .mat, {len(save_paths)} save_info')

download_files(MS_DATASET, mat_paths, MAT_LOCAL_DIR, revision=MS_REVISION, token=MS_TOKEN, verbose=True)
download_files(MS_DATASET, save_paths, SAVE_INFO_LOCAL, revision=MS_REVISION, token=MS_TOKEN, verbose=True)

from src.dataset import load_save_info_intensity
intensities = load_save_info_intensity(SAVE_INFO_LOCAL)
print(f'Intensity labels: {len(intensities)} (expect ~1600)')

run([PY, 'scripts/preprocess_to_npz.py',
     '--mat-dir', MAT_LOCAL_DIR, '--mat-pattern', '*.mat',
     '--save-info-dir', SAVE_INFO_LOCAL,
     '--output', NPZ_OUT,
     '--val-ratio', '0.1', '--test-ratio', '0.1',
     '--split-unit', 'trial', '--max-windows-per-trial', '40',
     '--tmp-dir', str(WORK / 'preprocessed')])

## 1.5 确保 npz 存在（本地或从 ModelScope 下载）

In [ ]:
from pathlib import Path
from src.ms_download import download_one_file, login_if_token

npz_local = Path(NPZ_OUT)
if not npz_local.exists():
    print(f'Downloading from {MS_DATASET}:{NPZ_REPO_PATH} ...')
    npz_local.parent.mkdir(parents=True, exist_ok=True)
    login_if_token(MS_TOKEN)
    dl = download_one_file(MS_DATASET, NPZ_REPO_PATH, str(npz_local.parent), revision=MS_REVISION, token=MS_TOKEN)
    dl_path = Path(dl)
    if dl_path.resolve() != npz_local.resolve() and dl_path.exists():
        dl_path.rename(npz_local)
    print(f'[OK] {NPZ_OUT} ({npz_local.stat().st_size / 1024**3:.2f} GB)')
else:
    print(f'[OK] {NPZ_OUT} ({npz_local.stat().st_size / 1024**3:.2f} GB)')

## 1.9 ⚡ OOM 终极修复：一次性转换 npz → mmap 格式

将压缩的 `seed_vii.npz` 拆分为：
- `seed_vii.npz.X.npy`：X 数组（未压缩，可内存映射，加载时 **零 RAM**）
- `seed_vii.npz.meta.npz`：y/s/meta/splits（很小）

**只需运行一次**，后续训练自动使用 mmap。转换需要一次性将 X 加载进内存。

In [ ]:
from src.dataset import ensure_mmap_format
x_npy, meta_npz = ensure_mmap_format(NPZ_OUT)
print(f'✓ X mmap: {x_npy}')
print(f'✓ Meta:   {meta_npz}')

# 验证 mmap 可用
import numpy as np
x_mmap = np.load(x_npy, mmap_mode='r')
print(f'✓ X shape={x_mmap.shape}, dtype={x_mmap.dtype} (mmap, 0 RAM)')
print(f'  单被试 ~{x_mmap.shape[1]*x_mmap.shape[2]*4*3200/1024**3:.2f} GB in RAM')

## 2. 训练（冻结 Intensity Head + 单被试训练）

训练脚本内部已实现 mmap 加载 + 子集拷贝，内存占用：
- 全量训练：~12GB（需要足够 RAM）
- 单被试训练：~600MB（**任何机器都能跑**）


In [ ]:
# ====== 构建训练命令 ======
train_cmd = [PY, 'scripts/train.py',
    '--data', NPZ_OUT,
    '--output-dir', RUN_DIR,
    '--device', 'auto', '--amp',
    '--pretrain-epochs', '5' if FREEZE_INTENSITY else '10',
    '--max-epochs', '100' if FREEZE_INTENSITY else '200',
    '--patience', '20' if FREEZE_INTENSITY else '30',
    '--batch-size', '64',
    '--num-workers', '2',
    '--lr', '2e-4', '--min-lr', '1e-5',
    '--sample-weight-mode', 'continuous',
    '--max-runtime-hours', '10',
]

if FREEZE_INTENSITY:
    train_cmd.append('--freeze-intensity-head')
    print('✓ 冻结 Intensity Head：只训练分类头')

if TRAIN_SUBJECTS:
    train_cmd.extend(['--train-subjects', TRAIN_SUBJECTS])
    print(f'✓ 训练被试: {TRAIN_SUBJECTS}')
if VAL_SUBJECTS:
    train_cmd.extend(['--val-subjects', VAL_SUBJECTS])
    print(f'✓ 验证被试: {VAL_SUBJECTS}')
if TEST_SUBJECTS:
    train_cmd.extend(['--test-subjects', TEST_SUBJECTS])
    print(f'✓ 测试被试: {TEST_SUBJECTS}')

print(f'\n训练命令: {" ".join(train_cmd)}')
run(train_cmd)

In [ ]:
# ====== 续训（取消注释即可）======
# run([PY, 'scripts/train.py',
#      '--data', NPZ_OUT,
#      '--output-dir', RUN_DIR,
#      '--device', 'auto', '--amp', '--resume',
#      '--freeze-intensity-head',
#      '--train-subjects', TRAIN_SUBJECTS,
#      '--val-subjects', VAL_SUBJECTS,
#      '--max-runtime-hours', '10'])

## 2.5 训练结果分析

In [ ]:
import json
from pathlib import Path

sp = Path(RUN_DIR) / 'summary.json'
if sp.exists():
    s = json.load(open(sp))
    print(f"Best Val Acc: {s['best_val_acc']:.4f} @{s['best_epoch']}")
    print(f"Epochs: {s['epochs_run']}, Params: {s['n_params']:,}")
    print(f"Freeze: {s['freeze_intensity_head']}, Train subj: {s['train_subjects']}")
    if s.get('test'): print(f"Test Acc: {s['test']['acc']:.4f}, MAE: {s['test']['intensity_mae']:.4f}")
else:
    print('No summary yet.')

## 3. 编码导出

In [ ]:
run([PY, 'scripts/encode.py',
     '--data', NPZ_OUT,
     '--checkpoint', str(pathlib.Path(RUN_DIR) / 'best_encoder.pt'),
     '--output', ENC_OUT,
     '--feature-type', 'projected',
     '--device', 'auto', '--amp'])